<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Máquinas de vectores de soporte (SVM)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🎨 SVM: clasificar el color de una imagen

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Identificar el color predominante de una imagen para adaptar el ambiente de una tienda de ropa. |
| 🔍 Comprensión de los datos | Descargamos 117 fotos etiquetadas en 9 colores y las leemos con OpenCV. |
| 🧹 Preparación de los datos | Redimensionamos a 100×100 px, aplanamos a 30 000 variables, escalamos a [0, 1] y dividimos 70/30. |
| 🤖 Modelado | Entrenamos una SVM lineal y buscamos `C` con validación cruzada (`GridSearchCV`). |
| 📊 Evaluación | Exactitud, informe por clase y matriz de confusión. |
| 🚀 Despliegue y conclusiones | Predecimos el color de fotos nuevas (carpeta `testimg`). |

> 🎯 **Analogía:** un decorador que mira una foto y decide en qué "familia de color" cae, trazando la frontera más ancha posible entre una familia y otra.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

---
## 🎯 Fase 1 · Comprensión del negocio

Identificar el color predominante de una imagen para adaptar el ambiente de una tienda de ropa.

📚 Lectura recomendada: [3]

## Análisis del problema

Una tienda de ropa quiere **cambiar el ambiente del local** (iluminación, decoración, exhibidores) según el **color predominante** de la mercadería. Para automatizarlo se entrena un modelo que, dada una foto, indique su color: negro, blanco, azul, verde, amarillo, naranja, rojo, café o violeta.

# Clasificar colores para cambiar el ambiente de un lugar de ventas de ropa

están propiamente relacionados con problemas de clasificación y regresión. Dado un conjunto de ejemplos de entrenamiento (de muestras) podemos etiquetar las clases y entrenar una SVM para construir un modelo que prediga la clase de una nueva muestra. Intuitivamente, una SVM es un modelo que representa a los puntos de muestra en el espacio, separando las clases a 2 espacios lo más amplios posibles mediante un hiperplano de separación definido como el vector entre los 2 puntos, de las 2 clases, más cercanos al que se llama vector soporte

- SVM

### Autores
   * Efrén Jiménez

### 🧪 Teoría rápida: ¿qué hace una SVM?

Una **máquina de vectores de soporte** busca el **hiperplano** que separa las clases con el **mayor margen** posible; los puntos más cercanos a la frontera son los *vectores de soporte* [3].

![SVM: margen máximo](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_09.png)
*Figura: de todos los hiperplanos posibles, la SVM elige el de máximo margen. [11]*

El parámetro **C** controla el equilibrio: un `C` grande castiga mucho los errores (margen estrecho, riesgo de sobreajuste); un `C` pequeño tolera errores (margen ancho).

![Efecto de C](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_10.png)
*Figura: efecto del parámetro C sobre el margen. [11]*

Si las clases no se separan con una recta, un **kernel** (p. ej. RBF) proyecta los datos a un espacio donde sí se pueden separar [4].

![Truco del kernel](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_13.png)
*Figura: el truco del kernel. [11]*

### Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import os
import pickle
import random
import pandas as pd
import sklearn
import scipy.io.wavfile as wav
from os import path
from sklearn.metrics import classification_report
from collections import defaultdict
from sklearn.metrics import accuracy_score
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns

import cv2

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import validation_curve
from sklearn import svm
from sklearn.svm import SVC

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

from sklearn.linear_model import LogisticRegression
import plotly.graph_objs as go

In [ ]:
import matplotlib.image as mpimg
from matplotlib.pyplot import imshow

def cv2_imshow(img):
    """Muestra una imagen leída con OpenCV (BGR) usando matplotlib; funciona en Colab y en local."""
    plt.figure(figsize=(3, 3)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()


---
## 🔍 Fase 2 · Comprensión de los datos — descargar y ver las imágenes

Descargamos 117 fotos etiquetadas en 9 colores y las leemos con OpenCV.

📚 Lectura recomendada: [7] [8]

In [ ]:
import urllib.request, zipfile

if not os.path.isdir('ColorClassification'):
    urllib.request.urlretrieve('https://github.com/ejimenezdelgado/ColorClassificationDataset/archive/refs/heads/master.zip', 'master.zip')
    with zipfile.ZipFile('master.zip') as z:
        z.extractall('.')
    with zipfile.ZipFile('ColorClassificationDataset-master/ColorClassification.zip') as z:
        z.extractall('.')

directory = "ColorClassification"
carpetas = sorted(f for f in os.listdir(directory) if f != 'testimg')
fig, axes = plt.subplots(1, len(carpetas), figsize=(2 * len(carpetas), 2.4))
for ax, c in zip(axes, carpetas):
    archivo = sorted(os.listdir(os.path.join(directory, c)))[0]
    ax.imshow(cv2.cvtColor(cv2.imread(os.path.join(directory, c, archivo)), cv2.COLOR_BGR2RGB))
    ax.set_title(c); ax.axis('off')
plt.show()

In [ ]:
datasetDirectory = "ColorClassification"

# Cada carpeta es un color (clase); se ordenan para que la etiqueta numérica sea siempre la misma
categories = sorted(f for f in os.listdir(datasetDirectory) if f != "testimg")
print(categories)

In [ ]:
img_size = 100

training_data = []
def create_training_data():
    for category in categories:
        path = os.path.join(datasetDirectory, category)
        class_num = categories.index(category)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img))
                new_array = cv2.resize(img_array, (img_size, img_size))
                training_data.append([new_array, class_num])
            except Exception as e:
                print("No se pudo leer", img, e)
create_training_data()
print("Imágenes cargadas:", len(training_data))

---
## 🔍 Fase 2 · Comprensión de los datos

Descargamos 117 fotos etiquetadas en 9 colores y las leemos con OpenCV.

## Entendimiento de los Datos

El total es de **117 imágenes** de **9 colores** (clases). Cada imagen se redimensiona a 100 × 100 píxeles con 3 canales (B, G, R), es decir **30 000 variables** numéricas continuas (0–255) y una etiqueta nominal (el color).

⚠️ Las clases están desbalanceadas (de 7 imágenes de *Green* a 20 de *Brown* y *Violet*), por eso conviene mirar métricas por clase y no solo la exactitud global.

In [ ]:
print('Etiqueta:', training_data[0][1], '| forma de la imagen:', training_data[0][0].shape)

In [ ]:
print(len(training_data), 'imágenes (cada una: matriz 100×100×3 + etiqueta)')

## Exploración de los Datos

---
## 🧹 Fase 3 · Preparación de los datos

Redimensionamos a 100×100 px, aplanamos a 30 000 variables, escalamos a [0, 1] y dividimos 70/30.

📚 Lectura recomendada: [5]

In [ ]:
X=[]
y=[]

for categories_x, label in training_data:
    X.append(categories_x)
    y.append(label)
X= np.array(X).reshape(len(training_data),-1)

In [ ]:
pd.Series([categories[c] for _, c in training_data], name='imágenes por color').value_counts().to_frame()

In [ ]:
X = X / 255.0          # escalar los píxeles a [0, 1]
y = np.array(y)
print(X.shape, y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                   test_size=0.30,
                                                   random_state=0)

---
## 🤖 Fase 4 · Modelado

Entrenamos una SVM lineal y buscamos `C` con validación cruzada (`GridSearchCV`).

📚 Lectura recomendada: [3] [4]

## Modelo de Machine Learning

In [ ]:

svc = SVC(kernel='linear',gamma='auto',C=2)
svc.fit(X_train, y_train)

In [ ]:
# Definir el rango de valores de C a probar
param_grid = {'C': [0.1, 1, 10, 100]}

# Crear una instancia del modelo de SVM
svc = SVC(kernel='linear', gamma='auto')

# Crear una instancia de GridSearchCV para validar diferentes valores de C
grid_search = GridSearchCV(svc, param_grid, cv=5)

# Entrenar el modelo con GridSearchCV
grid_search.fit(X_train, y_train)

# Imprimir los resultados de la validación cruzada
print("Mejor valor de C encontrado: ", grid_search.best_params_)
print("Exactitud promedio en validación cruzada: ", grid_search.best_score_)

In [ ]:
C_values = grid_search.cv_results_['param_C'].data.astype(float)
accuracy_values = grid_search.cv_results_['mean_test_score']

# Graficamos los resultados
plt.semilogx(C_values, accuracy_values)
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('Validation Curve with SVM')
plt.show()

In [ ]:
accuracy_values

In [ ]:
y_pred = grid_search.predict(X_test)

---
## 📊 Fase 5 · Evaluación

Exactitud, informe por clase y matriz de confusión.

📚 Lectura recomendada: [10]

## Evaluación

In [ ]:
from sklearn.metrics import accuracy_score
print("Accuracy on unknown data is",accuracy_score(y_test,y_pred))

In [ ]:
print(classification_report(y_test, y_pred, labels=range(len(categories)), target_names=categories, zero_division=0))

In [ ]:
# Grafica la matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=grid_search.classes_)

# Crea la visualización de la matriz de confusión
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=categories)

# Grafica la matriz de confusión
disp.plot(cmap=plt.cm.Blues)

plt.show()

---
## 🚀 Fase 6 · Despliegue y conclusiones — predecir el color de fotos nuevas

Predecimos el color de fotos nuevas (carpeta `testimg`).

📚 Lectura recomendada: [9]

In [ ]:
!wget https://upload.wikimedia.org/wikipedia/commons/thumb/6/67/Yellow-maple.jpg/800px-Yellow-maple.jpg

In [ ]:
# Predecir el color de fotos nuevas (carpeta testimg, que no se usó para entrenar)
def predecir_color(ruta):
    img = cv2.resize(cv2.imread(ruta), (img_size, img_size))
    x = img.reshape(1, -1) / 255.0
    return categories[int(grid_search.predict(x)[0])], img

ruta_test = os.path.join(datasetDirectory, 'testimg')
muestras = sorted(os.listdir(ruta_test))[:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for ax, nombre in zip(axes.ravel(), muestras):
    color, img = predecir_color(os.path.join(ruta_test, nombre))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title('Predicción: ' + color); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
result = pd.DataFrame({'original' : y_test,'predicted' : y_pred})

In [ ]:
result

---
## 🧪 Refuerzo: mejorar las características: de 30 000 píxeles a 18 rasgos de color

Usar los 30 000 valores de píxeles crudos es costoso y poco informativo: el color de una foto está resumido en **cuánta luz, qué tono y qué saturación** tiene. En el espacio **HSV** (*Hue*, *Saturation*, *Value*) el tono (*H*) separa los colores independientemente de la iluminación [7]. Resumimos cada imagen con la media y la desviación estándar de B, G, R, H, S, V (12 números) y un histograma de tono de 6 bandas, y comparamos con los píxeles crudos usando validación cruzada estratificada [5].

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC

def rasgos_color(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    canales = np.concatenate([img_bgr.reshape(-1, 3), hsv.reshape(-1, 3)], axis=1).astype(float)
    media, desv = canales.mean(axis=0), canales.std(axis=0)
    hist = cv2.calcHist([hsv], [0], None, [6], [0, 180]).ravel()
    return np.concatenate([media, desv, hist / hist.sum()])

X_rasgos = np.array([rasgos_color(img) for img, _ in training_data])
y_rasgos = np.array([clase for _, clase in training_data])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

resultados = {
    'Píxeles crudos (30 000) + SVM lineal': cross_val_score(SVC(kernel='linear', C=2), X, y, cv=cv).mean(),
    '18 rasgos de color + SVM lineal':      cross_val_score(make_pipeline(StandardScaler(), SVC(kernel='linear', C=2)), X_rasgos, y_rasgos, cv=cv).mean(),
    '18 rasgos de color + SVM RBF':         cross_val_score(make_pipeline(StandardScaler(), SVC(kernel='rbf', C=10, gamma='scale')), X_rasgos, y_rasgos, cv=cv).mean(),
}
pd.Series(resultados, name='Exactitud (5-fold)').round(3).to_frame()

### ✍️ Ejercicios
1. ¿Qué representación funcionó mejor y por qué? ¿Qué colores se confunden más (p. ej. *Brown* y *orange*)?
2. Pruebe `C` = 0.1, 1, 10, 100 y `gamma` = 0.01, 0.1, 1 con `GridSearchCV` sobre los 18 rasgos.
3. Agregue una foto propia (celular) a `ColorClassification/testimg` y prediga su color dominante.

---
## ✅ Conclusiones y siguientes pasos

La SVM con píxeles crudos funciona con pocos datos, pero es costosa (30 000 variables) y sensible a la iluminación. Resumir cada imagen con 18 rasgos de color (HSV) en lugar de 30 000 píxeles subió la exactitud (validación cruzada) de ≈ 0.79 a ≈ 0.89 con una fracción del cómputo [5]. Con más fotos por color, o con redes convolucionales, el desempeño sube aún más.

---
## 📚 Referencias

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*(3), 273–297. https://doi.org/10.1007/BF00994018

**[4]** Boser, B. E., Guyon, I. M., & Vapnik, V. N. (1992). A training algorithm for optimal margin classifiers. *Proceedings of the 5th Annual Workshop on Computational Learning Theory (COLT '92)*, 144–152. https://doi.org/10.1145/130385.130401

**[5]** Hsu, C.-W., Chang, C.-C., & Lin, C.-J. (2003). *A practical guide to support vector classification*. Department of Computer Science, National Taiwan University. https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf

**[6]** Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

**[7]** Bradski, G. (2000). The OpenCV Library. *Dr. Dobb's Journal of Software Tools*. https://opencv.org

**[8]** Jiménez Delgado, E. (s. f.). *ColorClassificationDataset* [Conjunto de datos de imágenes por color]. GitHub. https://github.com/ejimenezdelgado/ColorClassificationDataset

**[9]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[10]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[11]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)